# Analyse comparative : spaCy vs Snowball
**LO17 - TD3 : Lemmatisation et Indexation du Corpus**

Ce notebook compare les résultats de lemmatisation obtenus par :
- **spaCy** (`fr_core_news_sm`) : lemmatisation contextuelle
- **Snowball** (NLTK) : racinisation par règles

## 0. Imports et chargement des données

In [ ]:
import math

import matplotlib.pyplot as plt

# ── Chargement du fichier lemmes.tsv ──────────────────────────────────────────
CHEMIN_TSV = "outputs/lemmes.tsv"

donnees = []  # liste de (article_id, token, lemme_spacy, stemme_snowball)
with open(CHEMIN_TSV, encoding="utf-8") as f:
    lignes = f.readlines()

for ligne in lignes[1:]:
    ligne = ligne.strip()
    if not ligne:
        continue
    parties = ligne.split("\t")
    if len(parties) == 4:
        donnees.append(tuple(parties))

tokens = [d[1] for d in donnees]
spacys = [d[2] for d in donnees]
snowballs = [d[3] for d in donnees]
article_ids = [d[0] for d in donnees]

print(f"✓ {len(donnees)} lignes chargées")
print(f"  Exemple : {donnees[:3]}")

## 1. Statistiques générales

In [ ]:
def compter_uniques(lst):
    return len(set(lst))


def taux_reduction(tokens, formes):
    nb_t = compter_uniques(tokens)
    nb_f = compter_uniques(formes)
    return (nb_t - nb_f) / nb_t * 100 if nb_t else 0


def tokens_inchanges(tokens, formes):
    n = sum(1 for t, f in zip(tokens, formes, strict=True) if t == f)
    return n, n / len(tokens) * 100 if tokens else 0


nb_total = len(tokens)
nb_tok_uniques = compter_uniques(tokens)
nb_spacy_uniques = compter_uniques(spacys)
nb_snow_uniques = compter_uniques(snowballs)
taux_spacy = taux_reduction(tokens, spacys)
taux_snow = taux_reduction(tokens, snowballs)
inch_spacy, p_inch_spacy = tokens_inchanges(tokens, spacys)
inch_snow, p_inch_snow = tokens_inchanges(tokens, snowballs)

print("─" * 50)
print(f"{'Métrique':<40} {'spaCy':>6}  {'Snowball':>8}")
print("─" * 50)
print(f"{'Tokens total':<40} {nb_total:>6}  {nb_total:>8}")
print(f"{'Tokens uniques':<40} {nb_tok_uniques:>6}  {nb_tok_uniques:>8}")
msg = f"{'Formes uniques après normalisation':<40}"
msg += f" {nb_spacy_uniques:>6}  {nb_snow_uniques:>8}"
print(msg)
msg2 = f"{'Taux de réduction vocabulaire (%)':<40}"
msg2 += f" {taux_spacy:>5.1f}%  {taux_snow:>7.1f}%"
print(msg2)
print(f"{'Tokens inchangés':<40} {inch_spacy:>6}  {inch_snow:>8}")
msg3 = f"{'Tokens inchangés (%)':<40}"
msg3 += f" {p_inch_spacy:>5.1f}%  {p_inch_snow:>7.1f}%"
print(msg3)
print("─" * 50)

## 2. Réduction du vocabulaire

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Réduction du vocabulaire", fontsize=14, fontweight="bold")

# ── Graphique 1 : barres comparatives ─────────────────────────────────────────
categories = ["Tokens\nuniques", "Lemmes/Stems\nuniques"]
vals_spacy = [nb_tok_uniques, nb_spacy_uniques]
vals_snow = [nb_tok_uniques, nb_snow_uniques]

x = [0, 1]
largeur = 0.3
axes[0].bar(
    [i - largeur / 2 for i in x],
    vals_spacy,
    largeur,
    label="spaCy",
    color="#4C72B0",
)
axes[0].bar(
    [i + largeur / 2 for i in x],
    vals_snow,
    largeur,
    label="Snowball",
    color="#DD8452",
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].set_ylabel("Nombre de formes uniques")
axes[0].set_title("Formes uniques avant/après normalisation")
axes[0].legend()
for i, (vs, vn) in enumerate(zip(vals_spacy, vals_snow, strict=True)):
    axes[0].text(i - largeur / 2, vs + 1, str(vs), ha="center", fontsize=9)
    axes[0].text(i + largeur / 2, vn + 1, str(vn), ha="center", fontsize=9)

# ── Graphique 2 : taux de réduction ───────────────────────────────────────────
methodes = ["spaCy", "Snowball"]
taux = [taux_spacy, taux_snow]
couleurs = ["#4C72B0", "#DD8452"]
barres = axes[1].bar(methodes, taux, color=couleurs, width=0.4)
axes[1].set_ylabel("Taux de réduction (%)")
axes[1].set_title("Taux de réduction du vocabulaire")
axes[1].set_ylim(0, max(taux) * 1.3)
for barre, t in zip(barres, taux, strict=True):
    axes[1].text(
        barre.get_x() + barre.get_width() / 2,
        barre.get_height() + 0.3,
        f"{t:.1f}%",
        ha="center",
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig("outputs/plot_reduction.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Distribution des longueurs de formes

In [ ]:
long_tokens = [len(t) for t in tokens]
long_spacy = [len(s) for s in spacys]
long_snow = [len(s) for s in snowballs]


def moyenne(lst):
    return sum(lst) / len(lst) if lst else 0


def ecart_type(lst):
    m = moyenne(lst)
    return math.sqrt(sum((x - m) ** 2 for x in lst) / len(lst)) if lst else 0


msg1 = f"Longueur moyenne — tokens : {moyenne(long_tokens):.2f} "
msg1 += f"(σ={ecart_type(long_tokens):.2f})"
print(msg1)
msg2 = f"Longueur moyenne — spaCy  : {moyenne(long_spacy):.2f}  "
msg2 += f"(σ={ecart_type(long_spacy):.2f})"
print(msg2)
msg3 = f"Longueur moyenne — Snowball: {moyenne(long_snow):.2f}  "
msg3 += f"(σ={ecart_type(long_snow):.2f})"
print(msg3)

# ── Histogrammes superposés ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

max_len = max(max(long_tokens), max(long_spacy), max(long_snow))
bins = list(range(1, max_len + 2))


def histogramme(valeurs, bins):
    compte = [0] * (max_len + 1)
    for v in valeurs:
        if v <= max_len:
            compte[v] += 1
    return compte[1:]


x = list(range(1, max_len + 1))
ax.plot(
    x,
    histogramme(long_tokens, bins),
    label="Tokens originaux",
    color="gray",
    linewidth=2,
)
ax.plot(
    x,
    histogramme(long_spacy, bins),
    label="Lemmes spaCy",
    color="#4C72B0",
    linewidth=2,
)
ax.plot(
    x,
    histogramme(long_snow, bins),
    label="Stems Snowball",
    color="#DD8452",
    linewidth=2,
    linestyle="--",
)
ax.set_xlabel("Longueur du token")
ax.set_ylabel("Nombre d'occurrences")
ax.set_title("Distribution des longueurs de tokens")
ax.legend()
ax.set_xlim(1, 20)

## 4. Top 20 des formes les plus fréquentes

In [ ]:
def top_n(lst, n=20):
    compteur = {}
    for v in lst:
        compteur[v] = compteur.get(v, 0) + 1
    trie = sorted(compteur.items(), key=lambda x: x[1], reverse=True)
    return trie[:n]


top_spacy = top_n(spacys)
top_snow = top_n(snowballs)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Top 20 des formes les plus fréquentes", fontsize=14, fontweight="bold")

for ax, top, titre, couleur in [
    (axes[0], top_spacy, "spaCy (lemmes)", "#4C72B0"),
    (axes[1], top_snow, "Snowball (stems)", "#DD8452"),
]:
    mots = [t[0] for t in top]
    counts = [t[1] for t in top]
    y = list(range(len(mots)))
    ax.barh(y, counts, color=couleur, alpha=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(mots)
    ax.invert_yaxis()
    ax.set_xlabel("Fréquence")
    ax.set_title(titre)
    for i, c in enumerate(counts):
        ax.text(c + 0.2, i, str(c), va="center", fontsize=8)

plt.tight_layout()
plt.savefig("outputs/plot_top20.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Divergences entre spaCy et Snowball

In [ ]:
# ── Calcul des divergences ─────────────────────────────────────────────────────
vus = set()
divergences = []
for _, token, spacy, snowball in donnees:
    if token not in vus and spacy != snowball:
        divergences.append((token, spacy, snowball))
        vus.add(token)

nb_div = len(divergences)
pct_div = nb_div / nb_tok_uniques * 100
print(f"Divergences : {nb_div} / {nb_tok_uniques} tokens uniques ({pct_div:.1f}%)")

# ── Camembert concordance / divergence ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Divergences spaCy vs Snowball", fontsize=14, fontweight="bold")

nb_concord = nb_tok_uniques - nb_div
axes[0].pie(
    [nb_concord, nb_div],
    labels=[f"Concordance\n({nb_concord})", f"Divergence\n({nb_div})"],
    colors=["#55A868", "#C44E52"],
    autopct="%1.1f%%",
    startangle=90,
)
axes[0].set_title("Proportion de divergences (tokens uniques)")

# ── Tableau des 15 premières divergences ──────────────────────────────────────
axes[1].axis("off")
entetes = ["Token", "spaCy", "Snowball"]
lignes_tab = [[t, s, n] for t, s, n in divergences[:15]]
table = axes[1].table(
    cellText=lignes_tab,
    colLabels=entetes,
    loc="center",
    cellLoc="left",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)
# En-tête en gras
for j in range(3):
    table[0, j].set_facecolor("#4C72B0")
    table[0, j].set_text_props(color="white", fontweight="bold")
axes[1].set_title("15 premiers exemples de divergences")

plt.tight_layout()
plt.savefig("outputs/plot_divergences.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Tokens par document

In [ ]:
compteur_docs = {}
for aid in article_ids:
    compteur_docs[aid] = compteur_docs.get(aid, 0) + 1

docs_tries = sorted(compteur_docs.items(), key=lambda x: x[1], reverse=True)
ids = [d[0] for d in docs_tries]
counts = [d[1] for d in docs_tries]

fig, ax = plt.subplots(figsize=(max(8, len(ids) * 0.5), 5))
barres = ax.bar(ids, counts, color="#4C72B0", alpha=0.8)
ax.set_xlabel("Identifiant article")
ax.set_ylabel("Nombre de tokens")
ax.set_title("Nombre de tokens par document")
ax.tick_params(axis="x", rotation=45)
moy = moyenne(counts)
ax.axhline(
    moy, color="red", linestyle="--", linewidth=1.5, label=f"Moyenne ({moy:.0f})"
)
ax.legend()

plt.tight_layout()
plt.savefig("outputs/plot_tokens_par_doc.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Moyenne tokens/doc : {moy:.1f}")
print(f"Min : {min(counts)} (article {ids[counts.index(min(counts))]})")
print(f"Max : {max(counts)} (article {ids[counts.index(max(counts))]})")

## 7. Conclusion comparative

In [ ]:
# ── Radar / tableau récapitulatif ──────────────────────────────────────────────
criteres = [
    "Formes uniques produites",
    "Taux de réduction (%)",
    "Longueur moy. des formes",
    "Tokens inchangés (%)",
    "Divergences avec l'autre méthode (%)",
]
vals_spacy_recap = [
    nb_spacy_uniques,
    round(taux_spacy, 1),
    round(moyenne(long_spacy), 2),
    round(p_inch_spacy, 1),
    round(pct_div, 1),
]
vals_snow_recap = [
    nb_snow_uniques,
    round(taux_snow, 1),
    round(moyenne(long_snow), 2),
    round(p_inch_snow, 1),
    round(pct_div, 1),
]

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis("off")
entetes = ["Critère", "spaCy", "Snowball"]
lignes_recap = [
    [c, vs, vn]
    for c, vs, vn in zip(criteres, vals_spacy_recap, vals_snow_recap, strict=True)
]
table = ax.table(
    cellText=lignes_recap,
    colLabels=entetes,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)
for j in range(3):
    table[0, j].set_facecolor("#2d2d2d")
    table[0, j].set_text_props(color="white", fontweight="bold")
# Colonne spaCy en bleu clair
for i in range(1, len(criteres) + 1):
    table[i, 1].set_facecolor("#dce8f5")
    table[i, 2].set_facecolor("#fde8d8")

ax.set_title("Tableau récapitulatif comparatif", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("outputs/plot_recap.png", dpi=150, bbox_inches="tight")
plt.show()